# Ablation 2: Complementary Boundary Q1 Analysis

`ablation2_complementary_boundary.py`를 E316 프로젝트 안에서 노트북으로 실행하기 위한 정리본입니다.

이 노트북은 다음을 수행합니다.
- `GH-ANFIS(base_only)` vs `GH-ANFIS(full)` 비교
- base confidence 기준 Q1 bin(low-confidence boundary samples) 분석
- fold별 bin 집계와 dataset/overall 요약 CSV 저장


In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()


def _looks_like_project_root(path):
    markers = [
        'model.py',
        'data.py',
        'learning.py',
        'utils.py',
        'ablation2_complementary_boundary.py',
    ]
    return all((path / marker).exists() for marker in markers)


if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not _looks_like_project_root(PROJECT_ROOT):
    candidate_roots = [
        PROJECT_ROOT / '03_Research' / 'GH-ANFIS_E316',
        PROJECT_ROOT / '03_Research' / 'GH-ANFIS_exp',
    ]
    for cand in candidate_roots:
        if _looks_like_project_root(cand):
            PROJECT_ROOT = cand
            break
    else:
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if _looks_like_project_root(parent):
                PROJECT_ROOT = parent
                break

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

Project root: /home/harp3133t/Research/03_Research/GH-ANFIS_E404


In [2]:
import pandas as pd
from IPython.display import display

from ablation2_complementary_boundary import run_ablation

## Run Config

기본값은 E316에 복사해 둔 `no_mi` weight 기준입니다.
필요하면 아래 값만 수정한 뒤 실행하면 됩니다.

In [3]:
RUN_CONFIG = {
    'mode': 'no_mi',
    'weight_root': 'hyper_parameter/cv_weights',
    'data_root': 'data',
    'n_folds': 5,
    'seed': 42,
    'n_bins': 4,
    'boundary_bin': 1,
    'out_dir': 'output/ablation2_complementary_boundary',
    'summary_check_path': None,  # 기존 summary와 비교하려면 경로 문자열로 바꾸세요.
    'project_root': PROJECT_ROOT,
    'verbose': True,
}

RUN_CONFIG

{'mode': 'no_mi',
 'weight_root': 'hyper_parameter/cv_weights',
 'data_root': 'data',
 'n_folds': 5,
 'seed': 42,
 'n_bins': 4,
 'boundary_bin': 1,
 'out_dir': 'output/ablation2_complementary_boundary',
 'summary_check_path': None,
 'project_root': PosixPath('/home/harp3133t/Research/03_Research/GH-ANFIS_E404'),
 'verbose': True}

In [4]:
results = run_ablation(**RUN_CONFIG)

fold_df = results['fold_df']
fold_metrics_df = results['fold_metrics_df']
q1_by_dataset_df = results['q1_by_dataset_df']
q1_overall_df = results['q1_overall_df']
failed_folds_df = pd.DataFrame(results['failed_folds'])
output_paths = {k: str(v) for k, v in results['output_paths'].items()}

output_paths

[Config] mode=no_mi, n_folds=5, seed=42
[Config] n_bins=4, boundary_bin=1
[Path] weight_root=/home/harp3133t/Research/03_Research/GH-ANFIS_E404/hyper_parameter/cv_weights
[Path] data_root=/home/harp3133t/Research/03_Research/GH-ANFIS_E404/data
[Path] out_dir=/home/harp3133t/Research/03_Research/GH-ANFIS_E404/output/ablation2_complementary_boundary
[Device] cuda

[Dataset] Breast_Cancer_Wisconsin_(Original)
  -> source=maincode_loader:load_bcwd_data, n_samples=699, n_features=9, n_splits=5
  fold 01 ok | n_val=140 | base_acc=0.9643 | full_acc=0.9786 | q1_count=35 (0.250)
  fold 02 ok | n_val=140 | base_acc=0.9714 | full_acc=0.9643 | q1_count=35 (0.250)
  fold 03 ok | n_val=140 | base_acc=0.9786 | full_acc=0.9714 | q1_count=35 (0.250)
  fold 04 ok | n_val=140 | base_acc=0.9643 | full_acc=0.9786 | q1_count=35 (0.250)
  fold 05 ok | n_val=139 | base_acc=0.9784 | full_acc=0.9784 | q1_count=34 (0.245)

[Dataset] Vowel
  -> source=maincode_loader:load_vowel_data, n_samples=990, n_features=26,

{'fold_bin_counts_csv': '/home/harp3133t/Research/03_Research/GH-ANFIS_E404/output/ablation2_complementary_boundary/fold_bin_counts.csv',
 'q1_summary_by_dataset_csv': '/home/harp3133t/Research/03_Research/GH-ANFIS_E404/output/ablation2_complementary_boundary/q1_summary_by_dataset.csv',
 'q1_summary_overall_csv': '/home/harp3133t/Research/03_Research/GH-ANFIS_E404/output/ablation2_complementary_boundary/q1_summary_overall.csv',
 'run_config_json': '/home/harp3133t/Research/03_Research/GH-ANFIS_E404/output/ablation2_complementary_boundary/run_config.json'}

## Q1 Summary

In [5]:
display(q1_by_dataset_df)
display(q1_overall_df)

,dataset,boundary_bin,n_success_folds,n_failed_folds,n_expected_folds,q1_n_samples,wrong_to_right,right_to_wrong,stable_right,stable_wrong,net_gain,net_rate,base_acc_q1,full_acc_q1,help_rate_q1,harm_rate_q1,error_recovery_rate_q1,q1_ratio_mean,q1_ratio_std
0,Breast_Cancer_Wisconsin_(Original),1,5,0,5,174,4,2,155,13,2,0.011494,0.902299,0.913793,0.022989,0.011494,0.235294,0.248921,0.002158
1,Vowel,1,5,0,5,245,1,0,212,32,1,0.004082,0.865306,0.869388,0.004082,0.000000,0.030303,0.247475,0.000000
2,Spambase,1,5,0,5,1150,74,41,849,186,33,0.028696,0.773913,0.802609,0.064348,0.035652,0.284615,0.249946,0.000109
3,Gisette,1,5,0,5,1500,4,2,1361,133,2,0.001333,0.908667,0.910000,0.002667,0.001333,0.029197,0.250000,0.000000


,scope,q1_n_samples,wrong_to_right,right_to_wrong,stable_right,stable_wrong,net_gain,net_rate,base_acc_q1,full_acc_q1,help_rate_q1,harm_rate_q1,error_recovery_rate_q1,datasets_with_success
0,overall,3069,83,45,2577,364,38,0.012382,0.85435,0.866732,0.027045,0.014663,0.185682,4


## Fold-Level Detail

In [6]:
display(fold_metrics_df)
display(fold_df.head(20))

,dataset,fold,status,task_kind,base_acc,base_f1,full_acc,full_f1
0,Breast_Cancer_Wisconsin_(Original),1,ok,binary,0.964286,0.947368,0.978571,0.969072
1,Breast_Cancer_Wisconsin_(Original),2,ok,binary,0.971429,0.959184,0.964286,0.948454
2,Breast_Cancer_Wisconsin_(Original),3,ok,binary,0.978571,0.969697,0.971429,0.959184
3,Breast_Cancer_Wisconsin_(Original),4,ok,binary,0.964286,0.947368,0.978571,0.969072
4,Breast_Cancer_Wisconsin_(Original),5,ok,binary,0.978417,0.969072,0.978417,0.969072
5,Vowel,1,ok,multiclass,0.944444,0.943964,0.944444,0.943964
6,Vowel,2,ok,multiclass,0.964646,0.963622,0.969697,0.969049
7,Vowel,3,ok,multiclass,0.989899,0.989868,0.989899,0.989868
8,Vowel,4,ok,multiclass,0.974747,0.974491,0.974747,0.974491
9,Vowel,5,ok,multiclass,0.954545,0.953799,0.954545,0.953799


,dataset,dataset_key,mode,fold,task_kind,status,stage,error,n_val_samples,bin_id,...,wrong_to_right,right_to_wrong,stable_right,stable_wrong,net_gain,net_rate,base_acc_bin,full_acc_bin,error_recovery_rate_bin,boundary_conf_quantile
0,Breast_Cancer_Wisconsin_(Original),Breast_Cancer_Wisconsin__Original___no_mi,no_mi,1,binary,ok,evaluation,,140,1,...,2,0,30,3,2,0.057143,0.857143,0.914286,0.4,0.991606
1,Breast_Cancer_Wisconsin_(Original),Breast_Cancer_Wisconsin__Original___no_mi,no_mi,1,binary,ok,evaluation,,140,2,...,0,0,35,0,0,0.000000,1.000000,1.000000,NaN,0.991606
2,Breast_Cancer_Wisconsin_(Original),Breast_Cancer_Wisconsin__Original___no_mi,no_mi,1,binary,ok,evaluation,,140,3,...,0,0,35,0,0,0.000000,1.000000,1.000000,NaN,0.991606
3,Breast_Cancer_Wisconsin_(Original),Breast_Cancer_Wisconsin__Original___no_mi,no_mi,1,binary,ok,evaluation,,140,4,...,0,0,35,0,0,0.000000,1.000000,1.000000,NaN,0.991606
4,Breast_Cancer_Wisconsin_(Original),Breast_Cancer_Wisconsin__Original___no_mi,no_mi,2,binary,ok,evaluation,,140,1,...,0,1,32,2,-1,-0.028571,0.942857,0.914286,0.0,0.996758
5,Breast_Cancer_Wisconsin_(Original),Breast_Cancer_Wisconsin__Original___no_mi,no_mi,2,binary,ok,evaluation,,140,2,...,0,0,33,2,0,0.000000,0.942857,0.942857,0.0,0.996758
6,Breast_Cancer_Wisconsin_(Original),Breast_Cancer_Wisconsin__Original___no_mi,no_mi,2,binary,ok,evaluation,,140,3,...,0,0,35,0,0,0.000000,1.000000,1.000000,NaN,0.996758
7,Breast_Cancer_Wisconsin_(Original),Breast_Cancer_Wisconsin__Original___no_mi,no_mi,2,binary,ok,evaluation,,140,4,...,0,0,35,0,0,0.000000,1.000000,1.000000,NaN,0.996758
8,Breast_Cancer_Wisconsin_(Original),Breast_Cancer_Wisconsin__Original___no_mi,no_mi,3,binary,ok,evaluation,,140,1,...,0,1,31,3,-1,-0.028571,0.914286,0.885714,0.0,0.989968
9,Breast_Cancer_Wisconsin_(Original),Breast_Cancer_Wisconsin__Original___no_mi,no_mi,3,binary,ok,evaluation,,140,2,...,0,0,35,0,0,0.000000,1.000000,1.000000,NaN,0.989968


## Repro Check / Failures

In [7]:
results['repro_check']

{'summary_path': None,
 'available': False,
 'status': 'skipped',
 'reason': 'summary_check_disabled',
 'datasets': {}}

In [8]:
failed_folds_df if len(failed_folds_df) > 0 else 'No failed folds.'

'No failed folds.'